# Data Selection and Cleaning

## 1. Loading the Data
The required Python libraries were imported and the 5 datasets were loaded into the notebook.  

In [120]:
import pandas as pd
import numpy as np

In [121]:
# 1. 
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

- The datasets cover different dimensions of the "Sustainable Living Quality Index", including economic, health and education, infrastructure, environment, and governance.
- The missing value symbol `..` was treated as a missing value.

In [122]:
economic_data = pd.read_csv("../Dataset/economic_indicators.csv", na_values=[".."])
infrastructure_data = pd.read_csv('../Dataset/infrastructure_indicators.csv', na_values=[".."])
environmental_data = pd.read_csv('../Dataset/environment_indicators.csv', na_values=[".."])
health_education_data = pd.read_csv('../Dataset/health_education_indicators.csv', na_values=[".."])
governance_data = pd.read_csv('../Dataset/governance_indicators.csv', na_values=[".."])


- After loading the datasets, the shape of each dataset was checked.  

In [123]:
print("Economic:", economic_data.shape)
print("Health and Education:", health_education_data.shape)
print("Infrastructure:", infrastructure_data.shape)
print("Environment:", environmental_data.shape)
print("Governance:", governance_data.shape)

Economic: (656, 10)
Health and Education: (656, 10)
Infrastructure: (656, 10)
Environment: (656, 10)
Governance: (653, 9)


## 2. Choosing Relevant Variables
relevant variables were selected from each dataset. The variables were grouped into five main dimensions:
- Economic conditions
- Health and education
- Infrastructure and basic services
- Environmental sustainability
- Governance and stability

Not all variables were selected, and only the most recent data were used.


### Economic

In [155]:
# 1. Define indicators to keep
economic_selected = economic_data[['Country Name', 'Country Code', 'Series Name','2020 [YR2020]','2021 [YR2021]','2022 [YR2022]','2023 [YR2023]','2024 [YR2024]']].copy()

# Keep only selected variables
economic_selected = economic_selected[
    economic_selected['Series Name'].isin([
        'GDP per capita (current US$)',
        'Inflation, consumer prices (annual %)',
        'Unemployment, total (% of total labor force) (modeled ILO estimate)'
    ])
].copy()

# Convert values to numeric
year_columns = ['2020 [YR2020]', '2021 [YR2021]', '2022 [YR2022]', '2023 [YR2023]', '2024 [YR2024]']
for col in year_columns:
    economic_selected[col] = pd.to_numeric(economic_selected[col], errors='coerce')

# Use 2024 first, if missing use 2023
economic_selected['selected_value'] = (economic_selected['2024 [YR2024]']
    .fillna(economic_selected['2023 [YR2023]'])
    .fillna(economic_selected['2022 [YR2022]'])
    .fillna(economic_selected['2021 [YR2021]'])
    .fillna(economic_selected['2020 [YR2020]'])
)

# Convert to wide format
economic_final = economic_selected.pivot_table(
    index=['Country Name', 'Country Code'],
    columns='Series Name',
    values='selected_value',
    aggfunc='first'
).reset_index()

# Rename columns
economic_final = economic_final.rename(columns={
    'Country Name': 'Country',
    'GDP per capita (current US$)': 'gdp_per_capita_recent',
    'Inflation, consumer prices (annual %)': 'inflation_recent',
    'Unemployment, total (% of total labor force) (modeled ILO estimate)': 'unemployment_recent'
})

economic_final.columns.name = None

print(economic_final.shape)
economic_final.head(211)

(215, 5)


,Country,Country Code,gdp_per_capita_recent,inflation_recent,unemployment_recent
0,Afghanistan,AFG,413.757895,-6.601186,13.687
1,Albania,ALB,11377.775743,2.215874,10.689
2,Algeria,DZA,5752.990767,4.046115,11.655
3,American Samoa,ASM,18017.458938,NaN,NaN
4,Andorra,AND,49303.649167,NaN,NaN
5,Angola,AGO,2665.874448,28.240495,14.020
6,Antigua and Barbuda,ATG,23542.452695,6.198867,NaN
7,Argentina,ARG,13969.783660,219.883929,7.150
8,Armenia,ARM,8556.214070,0.269512,12.405
9,Aruba,ABW,39498.594129,NaN,NaN


- 2024 was used as the reference year. Where 2024 values were missing, 2023 values were used as a one-year backfill.

### Health and Education

In [141]:
health_education_selected = health_education_data[['Country Name', 'Country Code', 'Series Name','2020 [YR2020]','2021 [YR2021]','2022 [YR2022]','2023 [YR2023]','2024 [YR2024]']].copy()

health_education_selected = health_education_selected[
    health_education_selected['Series Name'].isin([
        'Life expectancy at birth, total (years)',
        'School enrollment, secondary (% gross)',
    ])
].copy()

# Convert values to numeric
year_columns = ['2020 [YR2020]','2021 [YR2021]','2022 [YR2022]','2023 [YR2023]','2024 [YR2024]']

for col in year_columns:
    health_education_selected[col] = pd.to_numeric(
        health_education_selected[col],
        errors='coerce'
    )

# Create selected value column
health_education_selected['selected_value'] = np.nan

# Life expectancy: use 2024 first, if missing use 2023
life_mask = health_education_selected['Series Name'] == 'Life expectancy at birth, total (years)'
health_education_selected.loc[life_mask, 'selected_value'] = (
    health_education_selected.loc[life_mask, '2024 [YR2024]']
    .fillna(health_education_selected.loc[life_mask, '2023 [YR2023]'])
)

# Secondary enrollment: latest available from 2024 to 2020
secondary_mask = health_education_selected['Series Name'] == 'School enrollment, secondary (% gross)'
health_education_selected.loc[secondary_mask, 'selected_value'] = (
    health_education_selected.loc[secondary_mask, '2024 [YR2024]']
    .fillna(health_education_selected.loc[secondary_mask, '2023 [YR2023]'])
    .fillna(health_education_selected.loc[secondary_mask, '2022 [YR2022]'])
    .fillna(health_education_selected.loc[secondary_mask, '2021 [YR2021]'])
    .fillna(health_education_selected.loc[secondary_mask, '2020 [YR2020]'])
)

# Pivot to wide format
health_education_final = health_education_selected.pivot_table(
    index=['Country Name', 'Country Code'],
    columns='Series Name',
    values='selected_value',
    aggfunc='first'
).reset_index()

health_education_final = health_education_final.rename(columns={
    'Country Name': 'Country',
    'Life expectancy at birth, total (years)': 'life_expectancy_2024',
    'School enrollment, secondary (% gross)': 'secondary_enrollment_recent',
})

health_education_final.columns.name = None

print(health_education_final.shape)
health_education_final.head(217)

(217, 4)


,Country,Country Code,life_expectancy_2024,secondary_enrollment_recent
0,Afghanistan,AFG,66.289000,59.613602
1,Albania,ALB,79.776000,108.355392
2,Algeria,DZA,76.475000,105.164132
3,American Samoa,ASM,72.992000,NaN
4,Andorra,AND,84.188000,101.923820
5,Angola,AGO,64.805000,51.483905
6,Antigua and Barbuda,ATG,77.766000,108.880779
7,Argentina,ARG,77.543000,105.574584
8,Armenia,ARM,78.319512,90.945871
9,Aruba,ABW,76.500000,124.379367


### Infrastructure

In [126]:
infrastructure_selected = infrastructure_data[['Country Name', 'Country Code', 'Series Name', '2023 [YR2023]', '2024 [YR2024]']].copy()

infrastructure_selected = infrastructure_selected[
    infrastructure_selected['Series Name'].isin([
        'Access to electricity (% of population)',
        'People using at least basic drinking water services (% of population)',
        'People using at least basic sanitation services (% of population)'
    ])
].copy()

infrastructure_selected['2023 [YR2023]'] = pd.to_numeric(infrastructure_selected['2023 [YR2023]'], errors='coerce')
infrastructure_selected['2024 [YR2024]'] = pd.to_numeric(infrastructure_selected['2024 [YR2024]'], errors='coerce')

infrastructure_selected['selected_value'] = infrastructure_selected['2024 [YR2024]'].fillna(infrastructure_selected['2023 [YR2023]'])

infrastructure_final = infrastructure_selected.pivot_table(
    index=['Country Name', 'Country Code'],
    columns='Series Name',
    values='selected_value',
    aggfunc='first'
).reset_index()

infrastructure_final = infrastructure_final.rename(columns={
    'Country Name': 'Country',
    'Access to electricity (% of population)': 'electricity_access_2024',
    'People using at least basic drinking water services (% of population)': 'drinking_water_access',
    'People using at least basic sanitation services (% of population)': 'sanitation_access'
})

infrastructure_final.columns.name = None

print(infrastructure_final.shape)
infrastructure_final.head(216)

(216, 5)


,Country,Country Code,electricity_access_2024,drinking_water_access,sanitation_access
0,Afghanistan,AFG,85.3,80.832437,54.495799
1,Albania,ALB,100.0,95.119427,99.299715
2,Algeria,DZA,100.0,92.420029,85.906486
3,American Samoa,ASM,NaN,100.000000,62.119698
4,Andorra,AND,100.0,100.000000,100.000000
5,Angola,AGO,51.1,67.960742,NaN
6,Antigua and Barbuda,ATG,100.0,98.922727,99.605719
7,Argentina,ARG,100.0,NaN,NaN
8,Armenia,ARM,100.0,99.931572,93.664943
9,Aruba,ABW,100.0,NaN,98.840002


### Environment

In [127]:
environment_selected = environmental_data[['Country Name', 'Country Code', 'Series Name', '2020 [YR2020]', '2021 [YR2021]', '2022 [YR2022]', '2023 [YR2023]']].copy()

environment_selected = environment_selected[
    environment_selected['Series Name'].isin([
        'Forest area (% of land area)',
        'PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)',
        'Renewable energy consumption (% of total final energy consumption)'
    ])
].copy()

# Convert values to numeric
environment_selected['2020 [YR2020]'] = pd.to_numeric(environment_selected['2020 [YR2020]'], errors='coerce')
environment_selected['2021 [YR2021]'] = pd.to_numeric(environment_selected['2021 [YR2021]'], errors='coerce')
environment_selected['2022 [YR2022]'] = pd.to_numeric(environment_selected['2022 [YR2022]'], errors='coerce')
environment_selected['2023 [YR2023]'] = pd.to_numeric(environment_selected['2023 [YR2023]'], errors='coerce')

# Create selected value column
environment_selected['selected_value'] = np.nan

# Forest area: use 2023 first, if missing use 2022
forest_mask = environment_selected['Series Name'] == 'Forest area (% of land area)'
environment_selected.loc[forest_mask, 'selected_value'] = (
    environment_selected.loc[forest_mask, '2023 [YR2023]']
    .fillna(environment_selected.loc[forest_mask, '2022 [YR2022]'])
)

# PM2.5: use 2020 because only 2020 data is available for most countries.
pm25_mask = environment_selected['Series Name'] == 'PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)'
environment_selected.loc[pm25_mask, 'selected_value'] = environment_selected.loc[pm25_mask, '2020 [YR2020]']

# Renewable energy: use 2022 first, if missing use 2021
renewable_mask = environment_selected['Series Name'] == 'Renewable energy consumption (% of total final energy consumption)'
environment_selected.loc[renewable_mask, 'selected_value'] = (
    environment_selected.loc[renewable_mask, '2022 [YR2022]']
    .fillna(environment_selected.loc[renewable_mask, '2021 [YR2021]'])
)

environment_final = environment_selected.pivot_table(
    index=['Country Name', 'Country Code'],
    columns='Series Name',
    values='selected_value',
    aggfunc='first'
).reset_index()

environment_final = environment_final.rename(columns={
    'Country Name': 'Country',
    'Forest area (% of land area)': 'forest_area_2023_reference',
    'PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)': 'pm2.5_exposure_2020',
    'Renewable energy consumption (% of total final energy consumption)': 'renewable_energy_recent'
})

environment_final.columns.name = None

print(environment_final.shape)
environment_final.head(215)

(215, 5)


,Country,Country Code,forest_area_2023_reference,pm2.5_exposure_2020,renewable_energy_recent
0,Afghanistan,AFG,1.852782,46.087094,20.0
1,Albania,ALB,28.791971,15.707004,41.9
2,Algeria,DZA,0.830314,25.552656,0.1
3,American Samoa,ASM,85.200000,6.715147,0.4
4,Andorra,AND,34.042553,9.080281,18.7
5,Angola,AGO,52.091270,25.145238,52.9
6,Antigua and Barbuda,ATG,18.013409,19.698270,0.9
7,Argentina,ARG,10.321295,14.908174,9.2
8,Armenia,ARM,11.618316,30.579633,9.1
9,Aruba,ABW,2.333333,NaN,8.8


### Governance

In [128]:
governance_selected = governance_data[['Country Name', 'Country Code', 'Series Name', '2023 [YR2023]', '2024 [YR2024]']].copy()

governance_selected = governance_selected[
    governance_selected['Series Name'].isin([
        'Government Effectiveness - Governance estimate (approx. -2.5 to +2.5)',
        'Rule of Law - Governance estimate (approx. -2.5 to +2.5)',
        'Political Stability - Governance estimate (approx. -2.5 to +2.5)'
    ])
].copy()

# Convert values to numeric
governance_selected['2023 [YR2023]'] = pd.to_numeric(governance_selected['2023 [YR2023]'], errors='coerce')
governance_selected['2024 [YR2024]'] = pd.to_numeric(governance_selected['2024 [YR2024]'], errors='coerce')
governance_selected['selected_value'] = governance_selected['2024 [YR2024]'].fillna(governance_selected['2023 [YR2023]'])

governance_final = governance_selected.pivot_table(
    index=['Country Name', 'Country Code'],
    columns='Series Name',
    values='selected_value',
    aggfunc='first'
).reset_index()

governance_final = governance_final.rename(columns={
    'Country Name': 'Country',
    'Government Effectiveness - Governance estimate (approx. -2.5 to +2.5)': 'government_effectiveness_2024',
    'Rule of Law - Governance estimate (approx. -2.5 to +2.5)': 'rule_of_law_2024',
    'Political Stability - Governance estimate (approx. -2.5 to +2.5)': 'political_stability_2024'
})

governance_final.columns.name = None

print(governance_final.shape)
governance_final.head()

(215, 5)


,Country,Country Code,government_effectiveness_2024,political_stability_2024,rule_of_law_2024
0,Afghanistan,AFG,-1.933997,-2.205211,-1.935489
1,Albania,ALB,0.312344,0.020467,-0.118502
2,Algeria,DZA,-0.251191,-0.693588,-0.696777
3,American Samoa,ASM,0.767382,1.253029,1.086581
4,Andorra,AND,1.159037,1.543796,1.409905
